# Stage 2: Two-Stage QLoRA Training with Unsloth
This notebook runs the two-stage QLoRA fine-tuning on a 3B parameter Bengali LLM. Stage 1 utilizes the broad mix of official and external dataset. Stage 2 anchors the model to the official data distribution only.

### 1. Install & Load Dependencies

In [ ]:
# Install training dependencies
%pip install -q -U unsloth xformers trl peft bitsandbytes accelerate datasets transformers matplotlib pandas

import os
import gc
import sys
import torch
import matplotlib.pyplot as plt
import pandas as pd
from datasets import Dataset
import datasets
datasets.disable_caching()
import datasets.arrow_dataset
datasets.arrow_dataset.generate_fingerprint = lambda *args, **kwargs: 'mock_fingerprint'
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

sys.path.append(os.path.abspath('src'))
from inference_utils import SYSTEM_PROMPT

print("PyTorch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

### 2. GPU Detection & Hyperparameter Scaling

In [ ]:
# Automatic batch size configuration based on GPU capabilities
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0.0

if vram_gb >= 20.0:
    # A100/H100 setup
    train_batch_size = 4
    gradient_accumulation_steps = 8
    print(f"Detected high VRAM GPU ({vram_gb:.2f} GB). Setting batch_size={train_batch_size}, grad_accum={gradient_accumulation_steps}")
else:
    # T4/L4 setup
    train_batch_size = 2
    gradient_accumulation_steps = 16
    print(f"Detected normal VRAM GPU ({vram_gb:.2f} GB). Setting batch_size={train_batch_size}, grad_accum={gradient_accumulation_steps}")

### 3. Load Base Model and Check Parameter Count

In [ ]:
BASE_MODEL = "hishab/titulm-llama-3.2-3b-v2.0"
FALLBACK_MODEL = "Kowshik24/Bangla-llama-3.2-3B-Instruct-QA-v2"
MAX_SEQ_LENGTH = 1024

try:
    print(f"Attempting to load primary base model: {BASE_MODEL}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )
except Exception as e:
    print(f"Primary model failed to load ({e}). Loading fallback model: {FALLBACK_MODEL}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=FALLBACK_MODEL,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )

# Verify the parameter count is under 3B limit
param_count = sum(p.numel() for p in model.parameters())
print(f"Base model parameter count (quantized representation): {param_count:,}")
if param_count > 3.0e9:
    raise ValueError(f"Model exceeds 3B parameter limit! Found: {param_count:,}")
else:
    print("PASS: Model size is compliant.")

### 4. Setup PEFT (QLoRA) Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("PEFT adapter configured successfully.")

### 5. Chat Template Formatting

In [ ]:
from unsloth.chat_templates import get_chat_template

try:
    # Map roles to the tokenizer chat template
    tokenizer = get_chat_template(
        tokenizer,
        chat_template="llama-3.2",
        mapping={"role": "role", "content": "content", "user": "user", "assistant": "assistant", "system": "system"},
        map_eos_token=True,
    )
    use_manual_template = False
    print("Successfully configured llama-3.2 chat template.")
except Exception as e:
    print(f"Warning: Failed to set chat template ({e}). Using manual Alpaca fallback.")
    use_manual_template = True

def format_dataset(df):
    texts = []
    for _, row in df.iterrows():
        inp, out = row["input"], row["output"]
        if not use_manual_template:
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": inp},
                {"role": "assistant", "content": out}
            ]
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        else:
            text = f"### System:\n{SYSTEM_PROMPT}\n\n### Instruction:\n{inp}\n\n### Response:\n{out}{tokenizer.eos_token}"
        texts.append(text)
    return Dataset.from_dict({"text": texts})

### 6. SFT Stage 1: Broad-Mix Training

In [ ]:
import inspect
import pandas as pd

# Load Stage 1 training data and validation data
stage1_df = pd.read_csv("working/train_plus_external_clean.csv")
val_df = pd.read_csv("working/sft_val.csv")

stage1_dataset = format_dataset(stage1_df)
print("Stage 1 Dataset Size:", len(stage1_dataset))

def build_sft_trainer(model, tokenizer, dataset, output_dir, lr, epochs, bs, grad_acc):
    major_cc = torch.cuda.get_device_capability()[0] if torch.cuda.is_available() else 0
    use_bf16 = torch.cuda.is_available() and major_cc >= 8 and torch.cuda.is_bf16_supported()
    use_fp16 = torch.cuda.is_available() and not use_bf16
    
    sft_args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=bs,
        gradient_accumulation_steps=grad_acc,
        warmup_ratio=0.03,
        num_train_epochs=epochs,
        learning_rate=lr,
        fp16=use_fp16,
        bf16=use_bf16,
        logging_steps=25,
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        report_to=[]
    )
    
    kwargs = {
        "model": model,
        "train_dataset": dataset,
        "dataset_text_field": "text",
        "max_seq_length": MAX_SEQ_LENGTH,
        "dataset_num_proc": 2,
        "packing": False,
        "args": sft_args,
    }
    sig = inspect.signature(SFTTrainer.__init__).parameters
    if "processing_class" in sig:
        kwargs["processing_class"] = tokenizer
    elif "tokenizer" in sig:
        kwargs["tokenizer"] = tokenizer
    return SFTTrainer(**kwargs)

# Stage 1 trainer config
trainer_stage1 = build_sft_trainer(
    model=model,
    tokenizer=tokenizer,
    dataset=stage1_dataset,
    output_dir="working/stage1_ckpt",
    lr=2e-4,
    epochs=1,
    bs=train_batch_size,
    grad_acc=gradient_accumulation_steps
)

# OOM Handling wrapper
try:
    print("Starting SFT Stage 1...")
    trainer_stage1.train()
except torch.cuda.OutOfMemoryError as e:
    print("Caught OOM during Stage 1. Clearing cache and halving batch size...")
    gc.collect()
    torch.cuda.empty_cache()
    # Halve batch size, double accumulation steps
    train_batch_size = max(1, train_batch_size // 2)
    gradient_accumulation_steps = gradient_accumulation_steps * 2
    trainer_stage1 = build_sft_trainer(
        model=model,
        tokenizer=tokenizer,
        dataset=stage1_dataset,
        output_dir="working/stage1_ckpt",
        lr=2e-4,
        epochs=1,
        bs=train_batch_size,
        grad_acc=gradient_accumulation_steps
    )
    trainer_stage1.train()

# Plot training loss curve
history = trainer_stage1.state.log_history
steps = [h["step"] for h in history if "loss" in h]
losses = [h["loss"] for h in history if "loss" in h]
plt.plot(steps, losses, label="Stage 1 Loss")
plt.title("Stage 1 Training Loss")
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.legend()
plt.show()

### 7. Evaluation of Model Output after Stage 1

In [ ]:
FastLanguageModel.for_inference(model)

print("=== Eval outputs after Stage 1 SFT ===")
for i in range(3):
    val_row = val_df.iloc[i]
    prompt = val_row["input"]
    ref = val_row["output"]
    
    if not use_manual_template:
        msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ]
        prompt_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    else:
        prompt_text = f"### System:\n{SYSTEM_PROMPT}\n\n### Instruction:\n{prompt}\n\n### Response:\n"
        
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.7, pad_token_id=tokenizer.eos_token_id)
    pred = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\nSample {i+1}:")
    print("Prompt:", prompt)
    print("Reference:", ref)
    print("Prediction:", pred.strip())

# Reset for training
FastLanguageModel.for_training(model)

### 8. SFT Stage 2: Official Anchor Training

In [ ]:
# Load Stage 2 training data (Official data only, excluding validation)
stage2_df = pd.read_csv("working/sft_train.csv")
stage2_dataset = format_dataset(stage2_df)
print("Stage 2 Dataset Size:", len(stage2_dataset))

# Stage 2 SFTTrainer
trainer_stage2 = build_sft_trainer(
    model=model,
    tokenizer=tokenizer,
    dataset=stage2_dataset,
    output_dir="working/stage2_ckpt",
    lr=5e-5,
    epochs=1,
    bs=train_batch_size,
    grad_acc=gradient_accumulation_steps
)

try:
    print("Starting SFT Stage 2...")
    trainer_stage2.train()
except torch.cuda.OutOfMemoryError as e:
    print("Caught OOM during Stage 2. Clearing cache and halving batch size...")
    gc.collect()
    torch.cuda.empty_cache()
    train_batch_size = max(1, train_batch_size // 2)
    gradient_accumulation_steps = gradient_accumulation_steps * 2
    trainer_stage2 = build_sft_trainer(
        model=model,
        tokenizer=tokenizer,
        dataset=stage2_dataset,
        output_dir="working/stage2_ckpt",
        lr=5e-5,
        epochs=1,
        bs=train_batch_size,
        grad_acc=gradient_accumulation_steps
    )
    trainer_stage2.train()

# Plot training loss curve for Stage 2
history2 = trainer_stage2.state.log_history
steps2 = [h["step"] for h in history2 if "loss" in h]
losses2 = [h["loss"] for h in history2 if "loss" in h]
plt.plot(steps2, losses2, label="Stage 2 Loss", color='orange')
plt.title("Stage 2 Training Loss")
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.legend()
plt.show()

### 9. Side-by-Side Validation Comparison

In [ ]:
FastLanguageModel.for_inference(model)

print("=== Eval outputs after Stage 2 SFT (Anchor) ===")
for i in range(3):
    val_row = val_df.iloc[i]
    prompt = val_row["input"]
    ref = val_row["output"]
    
    if not use_manual_template:
        msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ]
        prompt_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    else:
        prompt_text = f"### System:\n{SYSTEM_PROMPT}\n\n### Instruction:\n{prompt}\n\n### Response:\n"
        
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.7, pad_token_id=tokenizer.eos_token_id)
    pred = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\nSample {i+1} Comparison:")
    print("Prompt:   ", prompt)
    print("Reference:", ref)
    print("Anchor Prediction:", pred.strip())

### 10. Merge Weights and Save Final Model

In [ ]:
save_path = "working/final_model"
print(f"Saving merged 16bit model to {save_path}...")
model.save_pretrained_merged(save_path, tokenizer, save_method="merged_16bit")

# Clear GPU memory
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()

# Verify parameters size post-merge by reloading fresh using native Hugging Face
from transformers import AutoModelForCausalLM
merged_model = AutoModelForCausalLM.from_pretrained(save_path, torch_dtype=torch.bfloat16, device_map="cpu")
post_param_count = sum(p.numel() for p in merged_model.parameters())
print(f"Reloaded merged model parameter count: {post_param_count:,}")
assert post_param_count <= 3.0e9, f"Disqualification Warning: Post-merge parameter count is too high: {post_param_count:,}"
print("PASS: Merged model parameter count is compliant.")

### 11. Final Checklist & Statistics

In [ ]:
import os

# Print final disk size of the saved model (cross-platform)
print("\n=== Merged Model Disk Usage ===")
total_size = 0
for dirpath, dirnames, filenames in os.walk("working/final_model"):
    for f in filenames:
        fp = os.path.join(dirpath, f)
        total_size += os.path.getsize(fp)
size_gb = total_size / (1024**3)
print(f"working/final_model: {size_gb:.2f} GB")

print("Final Stage 1 Training Loss:", losses[-1] if losses else "N/A")
print("Final Stage 2 Training Loss:", losses2[-1] if losses2 else "N/A")